# motif-discover: Live Benchmark

This notebook runs **motif-discover** and **STREME** side-by-side on ENCODE K562 ChIP-seq data and visualizes the results in real time.

All computation happens live — no pre-computed results. Speed comparisons are relative to this machine's CPU.

## 1. Setup

In [ ]:
import os, subprocess

# Clone repo if not already present
if not os.path.isdir('/content/motif-discover/.git'):
    subprocess.run('rm -rf /content/motif-discover && git clone -q https://github.com/Travis42/motif-discover.git /content/motif-discover', shell=True)

os.chdir('/content/motif-discover')
subprocess.run(['chmod', '+x', 'motif-discover', 'streme'])

n_tfs = len([f for f in os.listdir('example') if f.endswith('.fa')])
print(f'Ready: {n_tfs} TFs in example/')
print(f'motif-discover: {os.path.getsize("motif-discover")/1024:.0f} KB')
print(f'STREME:         {os.path.getsize("streme")/1024:.0f} KB')

## 2. Run Live Benchmark

Runs both tools on all example TFs. motif-discover processes all TFs in ~4 seconds; STREME takes ~30+ seconds.

In [ ]:
%%time
import subprocess

result = subprocess.run(
    ['./motif-discover', '--data', 'example/', '--no-meme', '--streme-bin', './streme'],
    capture_output=True, text=True
)
print(result.stderr if result.returncode != 0 else 'Benchmark complete')
print(result.stdout)

## 3. Parse Results

In [ ]:
import pandas as pd

rows = []
for line in result.stdout.split('\n'):
    parts = line.split('\t')
    if len(parts) >= 6 and parts[0] not in ('TF', '') and parts[4] in ('ours', 'streme'):
        rows.append({
            'TF': parts[0],
            'Width': int(parts[1]),
            'AUROC': float(parts[2]),
            'Time_s': float(parts[3]),
            'Tool': parts[4],
        })

df = pd.DataFrame(rows)
ours = df[df['Tool'] == 'ours'].set_index('TF')
streme = df[df['Tool'] == 'streme'].set_index('TF')

common = ours.index.intersection(streme.index)
print(f'TFs benchmarked: {len(common)}')
print(f'\nmotif-discover:  AUROC={ours.loc[common, "AUROC"].mean():.4f}  ({ours.loc[common, "Time_s"].mean():.2f}s/TF)')
print(f'STREME:          AUROC={streme.loc[common, "AUROC"].mean():.4f}  ({streme.loc[common, "Time_s"].mean():.2f}s/TF)')
print(f'Speedup:         {streme.loc[common, "Time_s"].mean() / ours.loc[common, "Time_s"].mean():.1f}x')

## 4. Visualize

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

MD_COLOR = '#2166AC'
ST_COLOR = '#D6604D'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Per-TF AUROC comparison (grouped bars)
ax = axes[0]
tf_list = sorted(common)
x = np.arange(len(tf_list))
w = 0.35
ax.barh(x - w/2, ours.loc[tf_list, 'AUROC'], w, color=MD_COLOR, alpha=0.8, label='motif-discover')
ax.barh(x + w/2, streme.loc[tf_list, 'AUROC'], w, color=ST_COLOR, alpha=0.8, label='STREME')
ax.set_yticks(x)
ax.set_yticklabels(tf_list, fontsize=9)
ax.set_xlabel('AUROC')
ax.set_title('Per-TF AUROC')
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.3)
ax.legend(loc='lower right', fontsize=9)

# Panel B: Scatter — motif-discover vs STREME
ax = axes[1]
colors = np.where(ours.loc[common, 'AUROC'] > streme.loc[common, 'AUROC'], MD_COLOR, ST_COLOR)
ax.scatter(streme.loc[common, 'AUROC'], ours.loc[common, 'AUROC'], alpha=0.7, s=50, c=colors)
ax.plot([0.4, 1.0], [0.4, 1.0], 'k--', alpha=0.3, label='y = x')
ax.set_xlabel('STREME AUROC', color=ST_COLOR, fontweight='bold')
ax.set_ylabel('motif-discover AUROC', color=MD_COLOR, fontweight='bold')
ax.set_title(f'motif-discover vs STREME (n={len(common)})')
for tf in common:
    d = ours.loc[tf, 'AUROC'] - streme.loc[tf, 'AUROC']
    if abs(d) > 0.1:
        ax.annotate(tf, (streme.loc[tf, 'AUROC'], ours.loc[tf, 'AUROC']), fontsize=8)
ax.legend(loc='lower right', fontsize=9)

# Panel C: Speed comparison
ax = axes[2]
md_time = ours.loc[common, 'Time_s'].mean()
st_time = streme.loc[common, 'Time_s'].mean()
bars = ax.bar(['motif-discover', 'STREME'], [md_time, st_time], 
              color=[MD_COLOR, ST_COLOR], edgecolor='black', linewidth=0.5, width=0.5)
for bar, t in zip(bars, [md_time, st_time]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{t:.2f}s', ha='center', fontsize=12, fontweight='bold')
ax.set_ylabel('Time per TF (seconds)')
ax.set_title(f'Speed (this machine)')

plt.tight_layout()
plt.show()

## 5. Statistical Test

In [ ]:
from scipy.stats import wilcoxon

diffs = ours.loc[common, 'AUROC'].values - streme.loc[common, 'AUROC'].values
wins = (diffs > 0.001).sum()
losses = (diffs < -0.001).sum()
ties = len(diffs) - wins - losses

if len(diffs) >= 5:
    stat, p = wilcoxon(diffs)
    print(f'Wilcoxon signed-rank test (n={len(common)}):')
    print(f'  p-value   = {p:.2e}')
else:
    print(f'Too few TFs ({len(common)}) for Wilcoxon test. Use the full 132-TF benchmark.')

print(f'\n  motif-discover wins: {wins}')
print(f'  STREME wins:         {losses}')
print(f'  Ties:                {ties}')
print(f'  Mean Δ AUROC:        +{diffs.mean():.4f}')

---

**Note:** These results are from a small 11-TF sample on this machine's CPU. The full benchmark (132 ENCODE K562 TFs) and paper are at [github.com/Travis42/motif-discover](https://github.com/Travis42/motif-discover).